# RQ20 — Sequential Upstream-Prefix Dependency

**Research question:** Does downstream-wheel True-Gate ranking depend on whether the already selected upstream prefix is physically correct?

This notebook reuses Binding B03 / password 0528 as a controlled mechanistic perturbation experiment. It does **not** train a new True-Gate model. The frozen MAIN v8 runtime is scored unchanged on downstream W2 and W4 scans collected under deliberately correct and wrong upstream prefixes.

The normal MAIN v8 runtime intentionally rejects `binding_scan` metadata. RQ20 bypasses only that metadata-type guard and calls the same frozen candidate-scoring path directly. Model features, scalers, coefficients and wheel/direction handling are unchanged.

In [28]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys
import json
import pandas as pd
import numpy as np

MYDRIVE = Path("/content/drive/MyDrive")

result_candidates = [
    MYDRIVE / "Padlock_Reproduction_v1" / "results" / "20_RQ20_Sequential_Prefix_Dependency",
    MYDRIVE / "Padlock_Reproduction_v1" / "Padlock_Reproduction_v1" / "results" / "20_RQ20_Sequential_Prefix_Dependency",
]
RESULT_DIR = next((p for p in result_candidates if p.exists()), result_candidates[0])
RESULT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(RESULT_DIR))
import importlib
import RQ20_prefix_dependency as rq20
rq20 = importlib.reload(rq20)

REPRO_ROOT = rq20.locate_repro_root(MYDRIVE)
BINDING_ZIP = rq20.locate_binding_zip(MYDRIVE)
BINDING_DECISIONS = rq20.locate_binding_decision_dir(MYDRIVE)
WORK_DIR = Path("/content/rq20_work")

print("RQ20 module:", Path(rq20.__file__).resolve())
print("Result folder:", RESULT_DIR)
print("Reproduction code:", REPRO_ROOT)
print("Binding B03 archive:", BINDING_ZIP)
print("Binding B03 decisions:", BINDING_DECISIONS)
print(f"Archive size: {BINDING_ZIP.stat().st_size / 1e6:.1f} MB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RQ20 module: /content/drive/MyDrive/Padlock_Reproduction_v1/results/20_RQ20_Sequential_Prefix_Dependency/RQ20_prefix_dependency.py
Result folder: /content/drive/MyDrive/Padlock_Reproduction_v1/results/20_RQ20_Sequential_Prefix_Dependency
Reproduction code: /content/drive/MyDrive/Padlock_Reproduction_v1/Padlock_Reproduction_v1
Binding B03 archive: /content/drive/MyDrive/dataset/raw/Binding_B03_0528.zip
Binding B03 decisions: /content/drive/MyDrive/dataset/binding_scans/_progress/Binding_B03_0528
Archive size: 272.0 MB


## Controlled dataset recovery

The full Binding B03 ZIP is copied once from Drive to local Colab storage and extracted locally. All WAV/CSV reads then occur on local disk rather than through the Drive mount.

Expected design:

- Stage 1: W1 candidate sweep → probe W2;
- Stage 2: W1 fixed correct, W2 candidate sweep → probe W4;
- 3 repeats × 10 upstream candidates × 2 directions per stage;
- 120 decision scans / 1,200 WAV runs total.

In [29]:
dataset_root, decision_root = rq20.prepare_local_binding_data(
    BINDING_ZIP,
    BINDING_DECISIONS,
    WORK_DIR,
)

manifest = rq20.build_control_manifest(decision_root)
manifest.to_csv(RESULT_DIR / "RQ20_Control_Manifest.csv", index=False)

qc = (
    manifest.groupby(["stage", "probe_wheel", "prefix_correct", "direction"], as_index=False)
    .agg(
        n_decisions=("decision_key", "size"),
        n_upstream_digits=("upstream_digit", "nunique"),
        n_repeats=("replicate", "nunique"),
    )
)

display(qc)
print("Dataset root:", dataset_root)
print("Decision rows:", len(manifest))
print("Unique physical contexts:", manifest.groupby(["stage", "replicate", "upstream_digit"]).ngroups)

assert len(manifest) == 120
assert manifest.groupby("stage").size().eq(60).all()
assert manifest.groupby("stage")["prefix_correct"].sum().eq(6).all()
print("Binding B03 control-manifest check passed.")

Reusing local Binding B03 extraction
Binding B03 raw session: /content/rq20_work/Binding_B03_0528/Binding_B03_0528 (1200 runs)
Using existing local dataset shim: /content/rq20_work/Binding_B03_0528/_rq20_dataset_root/raw/Binding_B03_0528
Reusing local Binding B03 decision JSON files


,stage,probe_wheel,prefix_correct,direction,n_decisions,n_upstream_digits,n_repeats
0,1,2,0,CCW,27,9,3
1,1,2,0,CW,27,9,3
2,1,2,1,CCW,3,1,3
3,1,2,1,CW,3,1,3
4,2,4,0,CCW,27,9,3
5,2,4,0,CW,27,9,3
6,2,4,1,CCW,3,1,3
7,2,4,1,CW,3,1,3


Dataset root: /content/rq20_work/Binding_B03_0528/_rq20_dataset_root
Decision rows: 120
Unique physical contexts: 60
Binding B03 control-manifest check passed.


## Frozen MAIN v8 scoring

MAIN v8 is not refitted or tuned. Each Binding B03 scan is scored as a 10-candidate downstream True-Gate ranking problem.

The first run extracts/scorers 1,200 local WAVs and saves `RQ20_MAINv8_Direction_Scores.csv`. Later runs reuse that cache.

In [30]:
sys.path.insert(0, str(REPRO_ROOT / "src"))
import true_gate_inference

engine = true_gate_inference.TrueGateInferenceEngine(REPRO_ROOT / "model_assets")

print("Runtime model:", engine.model_version)
print("Source model SHA-256:", engine.source_model_sha256)

score_cache = RESULT_DIR / "RQ20_MAINv8_Direction_Scores.csv"
direction_scores = rq20.score_manifest_with_main_v8(
    manifest=manifest,
    dataset_root=dataset_root,
    engine=engine,
    cache_path=score_cache,
)

assert len(direction_scores) == 120
print("Direction-specific scoring complete.")

Runtime model: MAIN_v8_ACCURACY_ENSEMBLE_RUNTIME_v1
Source model SHA-256: bfffd49453315d7cb835a409105e4bf3a29412f6b5a51c75a9ac48d41f007e6a
Reusing cached MAIN v8 direction scores
Direction-specific scoring complete.


## Pair independently reseated directions

Binding B03 has one A recording per direction, not the deployed A+B structure. For each upstream candidate/repeat, CCW and CW scores are therefore averaged only as a **direction-paired diagnostic**.

This must not be reported as standard deployed MAIN v8 fusion performance.

In [31]:
paired = rq20.build_direction_paired_scores(direction_scores)
metrics, repeat_effects, candidate_summary, direction_summary = rq20.summarise_prefix_effects(
    paired,
    direction_scores,
)

rq20.save_tables(
    manifest,
    direction_scores,
    paired,
    metrics,
    repeat_effects,
    candidate_summary,
    direction_summary,
    RESULT_DIR,
)

paired_metrics = metrics[metrics["view"] == "direction-paired"].copy()
display(paired_metrics)
display(repeat_effects)

assert len(paired) == 60
assert paired.groupby("stage").size().eq(30).all()
assert paired.groupby("stage")["prefix_correct"].sum().eq(3).all()

,view,stage,probe_wheel,prefix_correct,n,top1,top2,top3,mean_true_rank,median_true_rank,mean_margin
0,direction-paired,1,2,0,27,0.629630,0.814815,0.851852,2.000000,1.0,0.555759
1,direction-paired,1,2,1,3,0.666667,0.666667,1.000000,1.666667,1.0,0.580259
2,direction-paired,2,4,0,27,0.666667,0.888889,1.000000,1.444444,1.0,0.573812
3,direction-paired,2,4,1,3,1.000000,1.000000,1.000000,1.000000,1.0,0.812362


,stage,replicate,probe_wheel,correct_upstream_digit,correct_true_rank,wrong_mean_true_rank,wrong_median_true_rank,wrong_minus_correct_mean_rank,correct_better_than_wrong_mean,correct_top1,wrong_top1_rate
0,1,1,2,0,3.0,2.555556,2.0,-0.444444,0,0,0.333333
1,1,2,2,0,1.0,2.444444,1.0,1.444444,1,1,0.555556
2,1,3,2,0,1.0,1.000000,1.0,0.000000,0,1,1.000000
3,2,1,4,5,1.0,1.333333,1.0,0.333333,1,1,0.777778
4,2,2,4,5,1.0,1.333333,1.0,0.333333,1,1,0.777778
5,2,3,4,5,1.0,1.666667,2.0,0.666667,1,1,0.444444


## Candidate-level and direction checks

A genuine sequential-state effect should not be inferred from one pooled number alone. The next tables retain:

- the true-rank profile for all ten upstream candidate digits;
- whether the physically correct upstream digit is consistently favourable across R1-R3;
- CCW/CW results separately.

In [32]:
display(candidate_summary.sort_values(["stage", "upstream_digit"]))
display(direction_summary.sort_values(["stage", "direction", "prefix_correct"]))

,stage,probe_wheel,upstream_digit,prefix_correct,n_repeats,mean_true_rank,sd_true_rank,top1,top2,top3
0,1,2,0,1,3,1.666667,1.154701,0.666667,0.666667,1.000000
1,1,2,1,0,3,1.333333,0.577350,0.666667,1.000000,1.000000
2,1,2,2,0,3,1.000000,0.000000,1.000000,1.000000,1.000000
3,1,2,3,0,3,3.333333,2.516611,0.333333,0.333333,0.666667
4,1,2,4,0,3,1.000000,0.000000,1.000000,1.000000,1.000000
5,1,2,5,0,3,1.333333,0.577350,0.666667,1.000000,1.000000
6,1,2,6,0,3,1.333333,0.577350,0.666667,1.000000,1.000000
7,1,2,7,0,3,3.333333,3.214550,0.333333,0.666667,0.666667
8,1,2,8,0,3,2.000000,1.732051,0.666667,0.666667,0.666667
9,1,2,9,0,3,3.333333,3.214550,0.333333,0.666667,0.666667


,stage,probe_wheel,direction,prefix_correct,n,top1,top2,top3,mean_true_rank
0,1,2,CCW,0,27,0.407407,0.592593,0.703704,2.925926
1,1,2,CCW,1,3,0.666667,0.666667,1.000000,1.666667
2,1,2,CW,0,27,0.592593,0.777778,0.851852,1.777778
3,1,2,CW,1,3,0.333333,1.000000,1.000000,1.666667
4,2,4,CCW,0,27,0.555556,0.777778,0.851852,2.074074
5,2,4,CCW,1,3,0.666667,1.000000,1.000000,1.333333
6,2,4,CW,0,27,0.629630,0.777778,0.888889,1.851852
7,2,4,CW,1,3,0.666667,1.000000,1.000000,1.333333


## Figures

RQ20 keeps two simple dissertation figures only:

- Figure 1: Correct vs wrong prefix Top-1 accuracy for Stage 1 and Stage 2.
- Figure 2: Correct vs wrong prefix mean true rank for Stage 1 and Stage 2.

Upstream-digit detail remains in the CSV tables and is not used as a main thesis figure.


In [ ]:
import importlib
rq20 = importlib.reload(rq20)

paired = pd.read_csv(RESULT_DIR / "RQ20_MAINv8_Direction_Paired.csv")
metrics = pd.read_csv(RESULT_DIR / "RQ20_Prefix_Metrics.csv")
candidate_summary = pd.read_csv(RESULT_DIR / "RQ20_Candidate_Summary.csv")

rq20.save_figures(
    paired=paired,
    metrics=metrics,
    candidate_summary=candidate_summary,
    result_dir=RESULT_DIR,
)


## Save report and conclusion

The report keeps Stage 1 and Stage 2 separate and explicitly records the small-n limitation. RQ20 is a controlled within-lock mechanistic experiment, not a cross-password population estimate.

In [34]:
rq20.write_report(
    metrics=metrics,
    repeat_effects=repeat_effects,
    candidate_summary=candidate_summary,
    direction_summary=direction_summary,
    result_dir=RESULT_DIR,
)

run_info = {
    "notebook": "20_RQ20_Sequential_Prefix_Dependency.ipynb",
    "research_question": "Does downstream-wheel True-Gate ranking depend on upstream prefix correctness?",
    "dataset": rq20.BINDING_SESSION,
    "password": rq20.EXPECTED_PASSWORD,
    "model": engine.model_version,
    "source_model_sha256": engine.source_model_sha256,
    "decision_scans": int(len(direction_scores)),
    "direction_paired_contexts": int(len(paired)),
    "stages": {
        "1": "sweep W1; probe W2",
        "2": "W1 fixed correct; sweep W2; probe W4",
    },
    "deployment_fusion": False,
    "diagnostic_fusion": "mean of independently reseated CCW and CW scores; one A recording per direction",
    "next_rq": "RQ21 Binding scan-level state detector / safety veto",
}

(RESULT_DIR / "RQ20_run_info.json").write_text(
    json.dumps(run_info, indent=2),
    encoding="utf-8",
)

print((RESULT_DIR / "RQ20_Report.md").read_text(encoding="utf-8"))
print("RQ20 outputs saved to:", RESULT_DIR)

# RQ20 — Sequential Upstream-Prefix Dependency

## Research question

Does downstream-wheel True-Gate ranking depend on whether the already selected upstream prefix is physically correct?

## Controlled dataset

Binding B03 (password 0528) is reused as a mechanistic perturbation experiment rather than as Binding-model training data.

- Stage 1: sweep W1 through all ten candidate digits and probe W2. The correct W1 digit is 0.
- Stage 2: hold W1 correct at 0, sweep W2 through all ten candidate digits and probe W4. The correct W2 digit is 5.
- Each upstream candidate is repeated in R1-R3 and recorded independently in CW and CCW after full release/reseat.
- MAIN v8 was trained only on the True-Gate B01-B07 development set and was not trained on Binding B03 / password 0528.
- Because Binding B03 contains only one A recording per direction, this RQ uses a direction-paired diagnostic average (CW + CCW), not the standard deployed A/B fusion protocol.

## Direction-paired results

| Stage | Pr